# 02. Foundation Model Benchmark Comparison & Analysis

This notebook aggregates and compares fine-tuning performance across foundation models:
- **TerraMind-1.0-base** (IBM / ESA)
- **Prithvi-EO-2.0-600M-TL** (IBM / NASA)
- **SatMAE++** (BiliSakura)
- **GFM Composition Pretraining** (05kashyap)

> **Note:** This notebook reads saved evaluation artifacts from `fine_tune/results/` and does NOT perform training.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "fine_tune" / "results"

models = ["terramind", "prithvi", "satmaepp", "gfm"]
records = []

for m in models:
    m_dir = RESULTS_DIR / m
    exp_json = m_dir / "experiment.json"
    metrics_json = m_dir / "metrics_valid.json"
    
    if not metrics_json.is_file():
        metrics_json = m_dir / "metrics_test.json"
        
    if exp_json.is_file():
        with open(exp_json, "r") as f:
            data = json.load(f)
        metrics = data.get("final_metrics", {})
        records.append({
            "Model": data.get("model_name", m),
            "Modality": data.get("modality", "-"),
            "Strategy": data.get("strategy", "-"),
            "Total Params (M)": data.get("total_parameters_million", 0.0),
            "Trainable Params (M)": data.get("trainable_parameters_million", 0.0),
            "Flood IoU": metrics.get("iou", np.nan),
            "Flood Dice": metrics.get("dice", np.nan),
            "Precision": metrics.get("precision", np.nan),
            "Recall": metrics.get("recall", np.nan),
            "Specificity": metrics.get("specificity", np.nan),
            "Accuracy": metrics.get("accuracy", np.nan),
            "Time (s)": data.get("training_time_seconds", np.nan),
        })
    elif metrics_json.is_file():
        with open(metrics_json, "r") as f:
            data = json.load(f)
        metrics = data.get("metrics", {})
        records.append({
            "Model": m,
            "Modality": "evaluated",
            "Strategy": "-",
            "Total Params (M)": 0.0,
            "Trainable Params (M)": 0.0,
            "Flood IoU": metrics.get("iou", np.nan),
            "Flood Dice": metrics.get("dice", np.nan),
            "Precision": metrics.get("precision", np.nan),
            "Recall": metrics.get("recall", np.nan),
            "Specificity": metrics.get("specificity", np.nan),
            "Accuracy": metrics.get("accuracy", np.nan),
            "Time (s)": np.nan,
        })

df = pd.DataFrame(records)
print("Foundation Model Comparison Table:")
display(df) if "display" in globals() else print(df.to_string())

## 2. Benchmark Metrics Comparison Chart

In [ ]:
if not df.empty and "Flood IoU" in df.columns:
    valid_df = df.dropna(subset=["Flood IoU"])
    if not valid_df.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(valid_df))
        width = 0.2
        
        ax.bar(x - 1.5*width, valid_df["Flood IoU"], width, label="Flood IoU", color="#1f77b4")
        ax.bar(x - 0.5*width, valid_df["Flood Dice"], width, label="Flood Dice / F1", color="#2ca02c")
        ax.bar(x + 0.5*width, valid_df["Precision"], width, label="Precision", color="#ff7f0e")
        ax.bar(x + 1.5*width, valid_df["Recall"], width, label="Recall", color="#d62728")
        
        ax.set_ylabel("Score [0.0 - 1.0]")
        ax.set_title("Geospatial Foundation Model Flood Segmentation Comparison")
        ax.set_xticks(x)
        ax.set_xticklabels(valid_df["Model"])
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

## 3. Training Convergence Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for m in models:
    log_csv = RESULTS_DIR / m / "training_log.csv"
    if log_csv.is_file():
        log_df = pd.read_csv(log_csv)
        if "epoch" in log_df and "val_iou" in log_df:
            ax.plot(log_df["epoch"], log_df["val_iou"], marker='o', label=f"{m.upper()} (Val IoU)")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Flood IoU")
ax.set_title("Validation IoU Progression over Epochs")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()